<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 1 &middot; Part 1 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Lab Setup: Python, NumPy and Jupyter

This first lab gets your tools working and refreshes the Python you will lean on for the
rest of the semester. Nothing here is machine learning yet &mdash; it is the vocabulary the
next three labs are written in.

**Companion theory lecture:** CSE 815, Week 1, Session 1 (Introduction to Machine Learning).

## What you will be able to do

1. Run, edit and restart a Jupyter notebook without losing your work.
2. Create NumPy arrays, inspect their `shape`, and index them.
3. Replace an explicit Python loop with a vectorised NumPy expression, and say why that matters.
4. Draw a labelled scatter plot and line plot with Matplotlib.

---

## 0. How a notebook works

A notebook is a sequence of **cells**. There are two kinds:

- **Markdown cells** (like this one) hold formatted text.
- **Code cells** hold Python. Running one sends it to the *kernel* &mdash; a live Python process
  that remembers everything you have run so far.

| Action | Keys |
|---|---|
| Run the cell, move to the next | `Shift` + `Enter` |
| Run the cell, stay put | `Ctrl` + `Enter` |
| Insert a cell below | `Esc` then `B` |
| Delete a cell | `Esc` then `D`, `D` |
| Change to markdown / code | `Esc` then `M` / `Y` |

> **The single most useful habit:** if results stop making sense, use
> *Kernel &rarr; Restart Kernel and Run All Cells*. Because the kernel remembers state, a
> variable you deleted from a cell can still be alive in memory, and a notebook that "works"
> can fail for anyone else. Restarting proves your notebook runs top to bottom.

## 1. Python refresher

Run each cell and read the output before moving on.

In [ ]:
# Variables have no type declaration; the type comes from the value.
course = "CSE 816"
credits = 1
alpha = 0.01
is_lab = True

print(course, type(course))
print(credits, type(credits))
print(alpha, type(alpha))
print(is_lab, type(is_lab))

In [ ]:
# f-strings are the readable way to build output. We use them constantly to report results.
w, b = 2.5, 1.0
print(f"model: f(x) = {w}x + {b}")
print(f"alpha = {alpha:.4f}")          # 4 decimal places
print(f"cost  = {1234.5678:8.2f}")     # width 8, 2 decimals -- keeps columns aligned

In [ ]:
# Lists are ordered and indexed from 0. Negative indices count from the end.
sizes = [0.8, 1.0, 1.2, 1.4, 1.6]

print("whole list :", sizes)
print("first      :", sizes[0])
print("last       :", sizes[-1])
print("first three:", sizes[0:3])      # 0, 1, 2 -- the endpoint is excluded
print("length     :", len(sizes))

In [ ]:
# Loops and conditionals. enumerate() gives you the index alongside the value.
for i, s in enumerate(sizes):
    label = "small" if s < 1.2 else "large"
    print(f"i = {i}, size = {s}, {label}")

In [ ]:
# Functions. The triple-quoted string is a docstring -- write one for every function you define.
def predict(x, w, b):
    """Return the value of the linear model w*x + b at the point x."""
    return w * x + b


print(predict(1.5, w=30.0, b=5.0))
print(predict(2.0, w=30.0, b=5.0))

### Checkpoint 1

In the empty cell below, write a function `average(values)` that returns the mean of a list
of numbers **using a loop** (no built-in `sum`). Test it on `sizes`.

*Expected:* `average(sizes)` should print `1.2`.

In [ ]:
# Your code here

---

## 2. NumPy: the array

A Python list can hold anything, which makes it flexible and slow. A NumPy **array** holds
one type in one contiguous block of memory, which makes it rigid and fast. Every data set,
every parameter vector and every prediction in this course is a NumPy array.

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)

In [ ]:
# Building arrays.
a = np.array([0.8, 1.0, 1.2, 1.4, 1.6])   # from a list
z = np.zeros(4)                            # four zeros
o = np.ones(3)                             # three ones
r = np.arange(0, 10, 2)                    # start, stop (excluded), step
l = np.linspace(0, 1, 5)                   # 5 points evenly spaced from 0 to 1 inclusive

print("a =", a)
print("z =", z)
print("o =", o)
print("r =", r)
print("l =", l)

### `shape` is the thing you will check most often

Almost every NumPy bug in this course is a shape mismatch. Get in the habit of printing
`.shape` whenever something surprises you.

`a.shape` is a **tuple**. For a one-dimensional array of length 5 it reads `(5,)` &mdash;
the trailing comma is Python's way of writing a one-element tuple, not a typo.

In [ ]:
print("a       :", a)
print("a.shape :", a.shape)       # (5,)  -> one dimension, 5 elements
print("a.ndim  :", a.ndim)        # number of dimensions
print("a.dtype :", a.dtype)       # element type

M = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
print()
print("M.shape :", M.shape)       # (2, 3) -> 2 rows, 3 columns
print("M[0, 2] :", M[0, 2])       # row 0, column 2
print("M[1]    :", M[1])          # the whole of row 1

In [ ]:
# Indexing and slicing work like lists, but slices are *views*, not copies.
print("a[0]    :", a[0])
print("a[-1]   :", a[-1])
print("a[1:4]  :", a[1:4])

view = a[1:4]
view[0] = 99.0            # this writes through to `a`!
print("a after modifying the slice:", a)

a[1] = 1.0                # put it back
print("restored          :", a)

---

## 3. Vectorisation

**Vectorisation** means expressing a computation over a whole array at once, instead of
looping over its elements in Python. NumPy pushes the loop down into compiled C code that
also uses your CPU's parallel (SIMD) instructions.

This is not a micro-optimisation. It is the difference between a model that trains in
seconds and one that trains overnight.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])
y = np.array([10.0, 20.0, 30.0, 40.0])

print("x + y  :", x + y)      # elementwise
print("x * y  :", x * y)      # elementwise -- NOT matrix multiplication
print("x ** 2 :", x ** 2)
print("2 * x  :", 2 * x)      # scalar is "broadcast" to every element
print("x - 1  :", x - 1)

### Broadcasting

When you combine an array with a scalar, NumPy *broadcasts* the scalar across every
element. That is why `2 * x` needs no loop. The same rule lets you write the whole
linear model `w * x + b` in one expression, for every training example at once.

In [ ]:
w = 30.0
b = 5.0

f = w * x + b        # one line, four predictions
print("x :", x)
print("f :", f)

### Reductions: collapsing an array to a number

In [ ]:
print("sum   :", np.sum(x))
print("mean  :", np.mean(x))
print("max   :", np.max(x))
print("argmax:", np.argmax(x))          # *index* of the maximum, not the value
print("dot   :", np.dot(x, y))          # sum of x[i]*y[i] -- the inner product
print("dot by hand:", np.sum(x * y))    # same thing, spelled out

### Loop versus vectorised: measure it

`%%timeit` is a notebook *magic*: it runs the cell many times and reports the average.
Compare the two cells below. The arrays are large enough for the difference to be obvious.

In [ ]:
big_x = np.random.rand(1_000_000)
big_y = np.random.rand(1_000_000)
print("array size:", big_x.shape)

In [ ]:
%%timeit
# The slow way: an explicit Python loop.
total = 0.0
for i in range(big_x.shape[0]):
    total += big_x[i] * big_y[i]

In [ ]:
%%timeit
# The fast way: one vectorised call.
total = np.dot(big_x, big_y)

You should see a factor of roughly **100&times; to 500&times;**. Both compute the same number;
only one of them is worth writing.

> **Rule for the rest of this course:** if you find yourself writing `for i in range(m)` over
> training examples, pause and ask whether NumPy can do it in one expression. In Part 2 you
> will write both versions of the model and check they agree.

### Checkpoint 2

Using the arrays `x` and `y` defined above, and **no Python loops**:

1. Compute the elementwise difference `x - y` and store it in `diff`.
2. Compute the sum of the squares of `diff`.
3. Divide that by `2 * len(x)`.

Print the final number. *(You have just computed a squared-error cost &mdash; the subject of
Part 3.)*

*Expected:* `303.75`

In [ ]:
# Your code here

---

## 4. Matplotlib

Every result in this course gets plotted. A number tells you *what*; a plot tells you *why*.

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")   # a clean default; harmless if unavailable

In [ ]:
# A scatter plot of some data, and a line through it.
x_data = np.array([0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.4])
y_data = np.array([30., 36., 45., 48., 58., 62., 70., 82.])

plt.figure(figsize=(6, 4))
plt.scatter(x_data, y_data, marker="o", c="#1B7F79", label="Training data")
plt.plot(x_data, 33.0 * x_data + 4.0, c="#9B2C4B", label="f(x) = 33x + 4")

plt.title("Flat price against floor area")
plt.xlabel("Size (1000 sq ft)")
plt.ylabel("Price (lakh BDT)")
plt.legend()
plt.show()

The four calls you will use over and over:

| Call | Does |
|---|---|
| `plt.scatter(x, y)` | individual points |
| `plt.plot(x, y)` | points joined by a line |
| `plt.xlabel` / `plt.ylabel` / `plt.title` | labels &mdash; an unlabelled axis costs marks |
| `plt.legend()` | shows the `label=` text of each series |

In [ ]:
# Two panels side by side, via subplots.
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))

ax[0].scatter(x_data, y_data, c="#1B7F79")
ax[0].set_title("The data")
ax[0].set_xlabel("Size (1000 sq ft)")
ax[0].set_ylabel("Price (lakh BDT)")

w_vals = np.linspace(0, 2, 100)
ax[1].plot(w_vals, (w_vals - 1.0) ** 2, c="#123B63")
ax[1].set_title("A cost-like curve")
ax[1].set_xlabel("w")
ax[1].set_ylabel("J(w)")

plt.tight_layout()
plt.show()

### Checkpoint 3

Plot the three lines `f(x) = 25x + 10`, `f(x) = 33x + 4` and `f(x) = 40x - 5` on top of a
scatter of `(x_data, y_data)`. Label the axes, give each line a `label=`, and add a legend.

Which of the three looks best to your eye? Write your answer in a markdown cell. In Part 3
you will replace this eyeballing with a number.

In [ ]:
# Your code here

---

## 5. Recap

- The kernel holds state; **Restart and Run All** is how you prove a notebook really works.
- `np.array` gives you fast, fixed-type arrays. Check `.shape` when confused.
- Vectorised expressions replace loops and run orders of magnitude faster.
- `w * x + b` on an array computes every prediction at once &mdash; this *is* the linear model.
- Label every axis.

### Exercises to hand in

1. Write `variance(v)` returning the population variance of a NumPy array, first with a
   loop and then vectorised. Verify both agree on `np.random.rand(50)` using
   `np.isclose`, and compare their timings with `%%timeit`.
2. Create `t = np.linspace(0, 4 * np.pi, 200)` and plot `np.sin(t)` and `np.sin(t) / (1 + t)`
   on the same axes, labelled and with a legend.
3. Let `A = np.arange(12).reshape(3, 4)`. Without running the code, predict `A.shape`,
   `A[1, 2]`, `A[:, 0]` and `A.sum(axis=0)`. Then run it and explain any prediction you got
   wrong. *(Hint: `axis=0` collapses rows.)*
4. Explain in two or three sentences why vectorisation is faster than a Python loop when
   both perform the same number of multiplications.

### Next

**Part 2 &mdash; Model Representation:** load a training set into NumPy, plot it, and implement
$f_{w,b}(x) = wx + b$.

**Reading:** James et al., *An Introduction to Statistical Learning*, 2nd ed., &sect;2.1 and &sect;3.1.